# H2O AutoML Pipeline

This notebook walks through a basic H2O AutoML workflow: loading data, splitting it, training multiple models automatically, and inspecting the leaderboard.

## 1. Initialize the H2O Cluster

In [1]:
import h2o
from h2o.automl import H2OAutoML

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Temurin-25.0.2+10 (build 25.0.2+10-LTS, mixed mode, sharing)
  Starting server from C:\Users\User\anaconda3\envs\cmi-flu-prediction-challenge-capstone\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\User\AppData\Local\Temp\tmpneg_kz3y
  JVM stdout: C:\Users\User\AppData\Local\Temp\tmpneg_kz3y\h2o_User_started_from_python.out
  JVM stderr: C:\Users\User\AppData\Local\Temp\tmpneg_kz3y\h2o_User_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,00 secs
H2O_cluster_timezone:,America/Los_Angeles
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 4 days
H2O_cluster_name:,H2O_from_python_User_krxn4c
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,15.41 Gb
H2O_cluster_total_cores:,16
H2O_cluster_allowed_cores:,16
H2O_cluster_status:,"locked, healthy"


## 2. Import the Data

In [2]:
data = h2o.import_file("merged_data/combined.parquet")

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


## 3. Identify Predictors and Response

In [3]:
x = [c for c in data.columns if not c.endswith('_d28') and not c.endswith('_d365')]
y = "HAI_Vic B/Austria/1359417/2021_d28"  # task 4.3
x = [c for c in x if c != y]

## 4. Convert Response to Factor (for Classification)

In [4]:
# data[y] = data[y].asfactor()

## 5. Split into Train and Test Sets

In [5]:
train, test = data.split_frame(ratios=[.8], seed=1234)

## 6. Run AutoML (20 Base Models, 1-Hour Cap)

In [6]:
aml = H2OAutoML(max_models=4, seed=1, max_runtime_secs=1200)
aml.train(x=x, y=y, training_frame=train)

AutoML progress: |
21:06:56.121: AutoML: XGBoost is not available; skipping it.

██
21:07:06.704: _train param, Dropping bad and constant columns: [TRAN_ENSG00000289025_d7, TRAN_ENSG00000294858_d7, TRAN_ENSG00000303447_d7, TRAN_ENSG00000225825_d7, TRAN_ENSG00000290014_d0, TRAN_ENSG00000303447_d0, TRAN_ENSG00000307809_d0, TRAN_ENSG00000290014_d7, TRAN_ENSG00000305628_d7, TRAN_ENSG00000289025_d0, TRAN_ENSG00000307809_d7, TRAN_ENSG00000305628_d0, TRAN_ENSG00000303459_d0, TRAN_ENSG00000224500_d7, TRAN_ENSG00000265046_d7, TRAN_ENSG00000302122_d0, TRAN_ENSG00000301266_d0, TRAN_ENSG00000302122_d7, TRAN_ENSG00000285900_d7, TRAN_ENSG00000265046_d0, TRAN_ENSG00000294858_d0, TRAN_ENSG00000301266_d7, TRAN_ENSG00000285900_d0, TRAN_ENSG00000304773_d0, TRAN_ENSG00000255256_d7, TRAN_ENSG00000303459_d7, TRAN_ENSG00000304773_d7, TRAN_ENSG00000306942_d7, TRAN_ENSG00000305616_d0, TRAN_ENSG00000164325_d7, TRAN_ENSG00000304761_d7, TRAN_ENSG00000304797_d0, TRAN_ENSG00000244128_d7, TRAN_ENSG00000306978_d0, TR

Model Details
=============
H2OGeneralizedLinearEstimator : Generalized Linear Modeling
Model Key: GLM_1_AutoML_1_20260416_210633


GLM Model: summary
    family    link      regularization               lambda_search                                                                number_of_predictors_total    number_of_active_predictors    number_of_iterations    training_frame
--  --------  --------  ---------------------------  ---------------------------------------------------------------------------  ----------------------------  -----------------------------  ----------------------  -----------------------------------------------
    gaussian  identity  Ridge ( lambda = 0.003381 )  nlambda = 30, lambda.max = 33.813, lambda.min = 0.003381, lambda.1se = -1.0  80309                         27848                          172                     AutoML_1_20260416_210633_training_py_2_sid_a61a

ModelMetricsRegressionGLM: glm
** Reported on train data. **

MSE: 2.119989969079925
RMSE: 1.456018533219933
MAE: 1.0630584383704724
RMSLE: 0.2580199234995463
Mean Residual Deviance: 2.119989969079925
R^2: 0.5274087704457876
Null degrees of freedom: 496
Residual degrees of freedom: -27352
Null deviance: 2229.4849094567344
Residual deviance: 1053.6350146327227
AIC: 57483.87634648745

ModelMetricsRegressionGLM: glm
** Reported on validation data. **

MSE: 1.9505691923166528
RMSE: 1.3966277930489042
MAE: 1.0808935310395207
RMSLE: 0.23747935655718763
Mean Residual Deviance: 1.9505691923166528
R^2: 0.5328630285154929
Null degrees of freedom: 80
Residual degrees of freedom: -27768
Null deviance: 346.13827073866946
Residual deviance: 157.99610457764888
AIC: 55983.985861480935

Scoring History: 
     timestamp            duration          iteration    lambda    predictors    deviance_train      deviance_test       alpha    iterations    training_rmse       training_deviance    training_mae        training_r2          validation_rmse     validation_deviance    validation_mae      validation_r2
---  -------------------  ----------------  -----------  --------  ------------  ------------------  ------------------  -------  ------------  ------------------  -------------------  ------------------  -------------------  ------------------  ---------------------  ------------------  ---------------------
     2026-04-16 21:07:27  0.000 sec         3            .34E2     27849         2.0589512452026746  4.213203427902529   0.0      3             2.0771816345264797  4.314683542814098    1.7624115211927984  0.03816450531565463  2.0526089223903634  4.213203388276529      1.6867158711403467  -0.009009615654924197
     2026-04-16 21:07:37  10.411 sec        5            .25E2     27849         2.0397557815341436  4.201753054453702   0.0
     2026-04-16 21:10:11  2 min 50.953 sec  6                                                                                     6             1.456018533219933   2.119989969079925    1.0630584383704724  0.5274087704457876   1.3966277930489042  1.9505691923166528     1.0808935310395207  0.5328630285154929
     2026-04-16 21:07:39  12.716 sec        8            .18E2     27849         2.017624847048334   4.188194818194948   0.0
     2026-04-16 21:07:42  15.254 sec        12           .13E2     27849         1.9923837594909344  4.171383128905899   0.0
     2026-04-16 21:07:44  17.260 sec        16           .95E1     27849         1.9638151241340407  4.147297521465753   0.0
     2026-04-16 21:07:46  19.720 sec        21           .69E1     27849         1.9310224718190547  4.105864943168078   0.0
     2026-04-16 21:07:49  22.582 sec        27           .5E1      27849         1.89307729367903    4.051418937250902   0.0
     2026-04-16 21:07:52  25.135 sec        32           .37E1     27849         1.8492663534467082  3.971130360872863   0.0
     2026-04-16 21:07:55  27.944 sec        38           .27E1     27849         1.797888033007288   3.8682722072649773  0.0
---  ---                  ---               ---          ---

## 7. View the AutoML Leaderboard

In [7]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

model_id                                                    rmse      mse      mae     rmsle    mean_residual_deviance
GLM_1_AutoML_1_20260416_210633                           1.39663  1.95057  1.08089  0.237479                   1.95057
StackedEnsemble_BestOfFamily_1_AutoML_1_20260416_210633  1.42476  2.02994  1.07821  0.243975                   2.02994
GBM_1_AutoML_1_20260416_210633                           1.44174  2.07861  1.08032  0.238918                   2.07861
StackedEnsemble_AllModels_1_AutoML_1_20260416_210633     1.44415  2.08558  1.08116  0.245872                   2.08558
DRF_1_AutoML_1_20260416_210633                           1.52728  2.33259  1.11429  0.25951                    2.33259
GBM_2_AutoML_1_20260416_210633                           1.5777   2.48914  1.16291  0.26694                    2.48914
[6 rows x 6 columns]



## 8. Inspect the Best Model

In [8]:
print(aml.leader)

Model Details
H2OGeneralizedLinearEstimator : Generalized Linear Modeling
Model Key: GLM_1_AutoML_1_20260416_210633


GLM Model: summary
    family    link      regularization               lambda_search                                                                number_of_predictors_total    number_of_active_predictors    number_of_iterations    training_frame
--  --------  --------  ---------------------------  ---------------------------------------------------------------------------  ----------------------------  -----------------------------  ----------------------  -----------------------------------------------
    gaussian  identity  Ridge ( lambda = 0.003381 )  nlambda = 30, lambda.max = 33.813, lambda.min = 0.003381, lambda.1se = -1.0  80309                         27848                          172                     AutoML_1_20260416_210633_training_py_2_sid_a61a

ModelMetricsRegressionGLM: glm
** Reported on train data. **

MSE: 2.119989969079925
RMSE: 1.4560185332199

In [9]:
predictions = aml.leader.predict(test)

results = test[y].cbind(predictions["predict"])
results.columns = ["actual", "predicted"]

# Filter out rows where actual is NaN
results_clean = results[results["actual"].isna() == 0]

print(results_clean.head(20))

glm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
  actual    predicted
 2.32193      3.68596
 3.32193      4.47015
 3.32193      2.87388
 5.32193      2.9021
 2.32193      3.80222
 6.32193      6.35741
 3.32193      3.42038
 2.32193      3.50779
 6.32193      6.11933
 3.32193      8.2963
 7.32193      3.53917
 6.32193      6.3522
 2.32193      3.81908
 5.32193      8.68305
 2.32193      4.04246
 2.32193      4.2304
 4.32193      6.68586
 2.32193      3.71001
 2.32193      2.6676
 6.32193      6.04062
[20 rows x 2 columns]



In [10]:
# # Save the model to disk
h2o.save_model(aml.leader, path="./model")
#
# # Load it back later
# saved_model = h2o.load_model("./my_model/model_id_here")

'C:\\Users\\User\\repos\\cmi-flu-prediction-challenge-capstone\\model\\GLM_1_AutoML_1_20260416_210633'